# Pollution Control

**Data Cleaning**

In [ ]:
#importing required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [55]:
#loading the data
df = pd.read_csv('../datasets/raw_cems_data.csv')
sensors = pd.read_csv('../datasets/sensor_master.csv')
manual = pd.read_csv("../datasets/manual_entries.csv")
threshold = pd.read_csv("../datasets/regulatory_thresholds.csv")
maint = pd.read_csv("../datasets/maintenance_logs.csv")

In [56]:
# Display first 5 records
df.head()

,Record_ID,Plant_ID,Stack_ID,Flow_Rate_m3_hr,TS,PM2.5,SO2,NOx,Unit,Status,Lat_Lon
0,E00001,PL-01,S-01,2033.5,01-01-2025 00:00,170.3,119.3,103.4,ug/m3,down,"13.083,80.2702"
1,E00002,PL-01,S-02,3377.9,01-01-2025 00:00,<2.0,94.2,31.6,µg/m³,MAINT,"13.0839,80.2721"
2,E00003,PL-02,S-01,3798.4,01-01-2025 00:00,123.9,NaN,101.3,mg/Nm3,FAULT,"19.0763,72.8777"
3,E00004,PL-03,S-01,3121.2,01-01-2025 00:00,117.8,147.1,100.8,ug/m3,OK,"12.9719,77.5955"
4,E00005,PL-03,S-02,3700.5,31-12-2024 24:00,37.0,15.0,157.1,mg/Nm3,FAULT,"12.9722,77.5941"


In [57]:
#checking the columns and its datatypes
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27121 entries, 0 to 27120
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Record_ID        27121 non-null  str    
 1   Plant_ID         27121 non-null  str    
 2   Stack_ID         27121 non-null  str    
 3   Flow_Rate_m3_hr  21682 non-null  float64
 4   TS               27121 non-null  str    
 5   PM2.5            26726 non-null  str    
 6   SO2              26705 non-null  str    
 7   NOx              26701 non-null  str    
 8   Unit             27121 non-null  str    
 9   Status           26999 non-null  str    
 10  Lat_Lon          27121 non-null  str    
dtypes: float64(1), str(10)
memory usage: 2.3 MB


In [58]:
#checking null values
df.isnull().sum()

Record_ID             0
Plant_ID              0
Stack_ID              0
Flow_Rate_m3_hr    5439
TS                    0
PM2.5               395
SO2                 416
NOx                 420
Unit                  0
Status              122
Lat_Lon               0
dtype: int64

In [59]:
audit_log = []


In [60]:
# C-1. Datetime normalization (24:10 → next day 00:10).
from datetime import datetime, timedelta

def clean_timestamp(df):
    df = df.copy()

    count_slash = 0
    count_iso = 0
    count_midnight = 0

    for i in df.index:
        ts = str(df.at[i, 'TS']).strip()

        # 1. fix slash (/ → -)
        if '/' in ts:
            ts = ts.replace('/', '-')
            count_slash += 1

        # 2. fix ISO format (YYYY-MM-DD → DD-MM-YYYY)
        if ts[:4].isdigit() and ts[4] == '-':
            try:
                dt = datetime.strptime(ts, '%Y-%m-%d %H:%M:%S')
                ts = dt.strftime('%d-%m-%Y %H:%M')
                count_iso += 1
            except:
                pass

        # 3. fix 24:00 → next day 00:00
        if '24:' in ts:
            try:
                parts = ts.split(' ')
                date_part = parts[0]
                minute = parts[1].split(':')[1]

                dt = datetime.strptime(date_part, '%d-%m-%Y') + timedelta(days=1)
                ts = dt.strftime('%d-%m-%Y') + f' 00:{minute}'

                count_midnight += 1
            except:
                pass

        df.at[i, 'TS'] = ts

    print("Slash fixed:", count_slash)
    print("ISO fixed:", count_iso)
    print("Midnight fixed:", count_midnight)

    return df

In [61]:
# df = clean_timestamp(df)

In [62]:
# C-17. Standard time base conversion to IST.
from datetime import datetime, timedelta

def convert_utc_to_ist(df):
    df = df.copy()

    count = 0

    for i in df.index:
        ts = str(df.at[i, 'TS'])

        if 'UTC' in ts:
            try:
                # remove UTC
                ts_clean = ts.replace('UTC', '').strip()

                # convert to datetime
                dt = datetime.strptime(ts_clean, '%d-%m-%Y %H:%M')

                # add 5:30 hours
                dt = dt + timedelta(hours=5, minutes=30)

                # convert back to string
                new_ts = dt.strftime('%d-%m-%Y %H:%M')

                # update dataframe
                df.at[i, 'TS'] = new_ts

                count += 1
            except:
                pass

    print("UTC converted to IST:", count)

    return df

In [63]:
# df = convert_utc_to_ist(df)

# df['TS'] = pd.to_datetime(df['TS'], dayfirst=True)

In [64]:
# C-2. Unit standardization (ug/m3, µg/m³ → ug/m3).
#def clean_unit(df):
    # convert to string and lowercase
 #   df["Unit"] = df["Unit"].astype(str).str.lower()
    # replace special characters
  #  df["Unit"] = df["Unit"].str.replace("³", "3", regex=False)
   # df["Unit"] = df["Unit"].str.replace("µ", "u", regex=False)
    #return df
def clean_unit(df):
    global audit_log

    # convert to string and lowercase
    df["Unit"] = df["Unit"].astype(str)

    for idx in df.index:
        old = df.at[idx, "Unit"]

        # apply transformations
        new = old.lower()
        new = new.replace("³", "3")
        new = new.replace("µ", "u")

        # log ONLY if changed
        if old != new:
            audit_log.append({
                'Record_ID': df.at[idx, 'Record_ID'],
                'Column': 'Unit',
                'Old': old,
                'New': new,
                'Rule': 'R2'
            })

        # assign back
        df.at[idx, "Unit"] = new

    return df

In [65]:
# df = clean_unit(df)

In [66]:
# C-3. Whitespace trimming and casing standardization for status.
#def clean_status(df):
 #   df = df.copy()

  #  df['Status'] = df['Status'].astype(str).str.strip().str.upper()

    # fix variations
   # df['Status'] = df['Status'].replace({
    #    'MAINTENANCE': 'MAINT',
     #   'ERROR 404': 'ERROR'
    #})

    #
    # return df
def clean_status(df):
    global audit_log

    df = df.copy()

    for idx in df.index:
        old = df.at[idx, 'Status']

        # apply cleaning
        new = str(old).strip().upper()

        # fix variations
        if new == 'MAINTENANCE':
            new = 'MAINT'
        elif new == 'ERROR 404':
            new = 'ERROR'

        # log only if changed
        if old != new:
            audit_log.append({
                'Record_ID': df.at[idx, 'Record_ID'],
                'Column': 'Status',
                'Old': old,
                'New': new,
                'Rule': 'R3'
            })

        # assign back
        df.at[idx, 'Status'] = new

    return df

In [67]:
# df = clean_status(df)

In [68]:
# C-4. Coordinate parsing and range validation.


def clean_coordinates(df):

    # Step 1: clean
    df["Lat_Lon"] = df["Lat_Lon"].astype(str).str.strip()

    # Step 2: split into Lat & Lon
    df[["Lat", "Lon"]] = df["Lat_Lon"].str.split(",", n=1, expand=True)

    # Step 3: convert to numeric safely
    df["Lat"] = pd.to_numeric(df["Lat"], errors="coerce")
    df["Lon"] = pd.to_numeric(df["Lon"], errors="coerce")

    # Step 4: range validation
    df.loc[~df["Lat"].between(-90, 90), "Lat"] = np.nan
    df.loc[~df["Lon"].between(-180, 180), "Lon"] = np.nan

    # Step 5: fill missing using Plant_ID mean
    df["Lat"] = df["Lat"].fillna(df.groupby("Plant_ID")["Lat"].transform("mean"))
    df["Lon"] = df["Lon"].fillna(df.groupby("Plant_ID")["Lon"].transform("mean"))



    return df

In [69]:
# df = clean_coordinates(df)

In [70]:
# C-5. Negative and impossible concentration checks
#def clean_pollutants(df):

 #   pollutant_cols = ["PM2.5", "SO2", "NOx"]

    # Step 1: convert to numeric
  #  df[pollutant_cols] = df[pollutant_cols].apply(pd.to_numeric, errors="coerce")

    # Step 2: replace negatives with NaN
   # df[pollutant_cols] = df[pollutant_cols].mask(df[pollutant_cols] < 0)

    # Step 3: fill missing using group mean
    #df[pollutant_cols] = df.groupby("Plant_ID")[pollutant_cols].transform(
     #   lambda x: x.fillna(x.mean())
    #)

    #return df
def clean_pollutants(df):
    global audit_log

    pollutant_cols = ["PM2.5", "SO2", "NOx"]

    # Step 1: convert to numeric
    df[pollutant_cols] = df[pollutant_cols].apply(pd.to_numeric, errors="coerce")

    # Step 2: replace negatives with NaN (LOG THIS)
    for col in pollutant_cols:
        for idx in df.index:
            val = df.at[idx, col]

            if pd.notna(val) and val < 0:
                audit_log.append({
                    'Record_ID': df.at[idx, 'Record_ID'],
                    'Column': col,
                    'Old': val,
                    'New': None,
                    'Rule': 'R5'
                })

                df.at[idx, col] = None

    # Step 3: fill missing using group mean (LOG THIS)
    for col in pollutant_cols:
        for plant, group in df.groupby("Plant_ID"):
            mean_val = group[col].mean()

            for idx in group.index:
                if pd.isna(df.at[idx, col]) and pd.notna(mean_val):
                    audit_log.append({
                        'Record_ID': df.at[idx, 'Record_ID'],
                        'Column': col,
                        'Old': None,
                        'New': mean_val,
                        'Rule': 'R5'
                    })

                    df.at[idx, col] = mean_val

    return df


In [71]:
# df = clean_pollutants(df)

In [72]:
# C-6. Duplicate merge by (Plant_ID, TS).
#def remove_duplicates(df):

 #   print(f'BEFORE — Total rows: {len(df)}')

    # remove duplicates
  #  before = len(df)
   # df = df.drop_duplicates(subset=['Plant_ID', 'Stack_ID', 'TS'], keep='first')
    #dupes_removed = before - len(df)

    # after check
    #print(f'AFTER  — Total rows: {len(df)}')
    #print(f'Duplicates removed: {dupes_removed}')

    #return df
def remove_duplicates(df):
    global audit_log

    print(f'BEFORE — Total rows: {len(df)}')

    # identify duplicates (before dropping)
    dup_mask = df.duplicated(subset=['Plant_ID', 'Stack_ID', 'TS'], keep='first')

    # log duplicates that will be removed
    for idx in df[dup_mask].index:
        audit_log.append({
            'Record_ID': df.at[idx, 'Record_ID'],
            'Column': 'ROW',
            'Old': 'Duplicate',
            'New': 'Removed',
            'Rule': 'R6'
        })

    # remove duplicates
    before = len(df)
    df = df[~dup_mask]
    dupes_removed = before - len(df)

    # after check
    print(f'AFTER  — Total rows: {len(df)}')
    print(f'Duplicates removed: {dupes_removed}')

    return df

In [73]:
# df = remove_duplicates(df)

In [74]:
# C-7. Sensor span/zero calibration application.
def apply_calibration(df, sensor):

    pollutant_cols = ['PM2.5', 'SO2', 'NOx']

    # merge calibration data
    df = df.merge(sensor, on=['Plant_ID', 'Stack_ID'], how='left')

    # apply calibration
    for col in pollutant_cols:
        df[col] = (df[col] - df['Zero_Drift']) / df['Span_Mult']

    return df

In [75]:
# df = apply_calibration(df,sensors)

In [76]:
#C-8. Outlier filtering using rolling median and Hampel filters.

def clean_outliers(df):
    df = df.copy()

    for col in ['PM2.5', 'SO2', 'NOx']:

        # 🔥 CRITICAL FIX: convert to numeric FIRST
        df[col] = pd.to_numeric(df[col], errors='coerce')

        spike_mask = df[col] > 5000
        count = spike_mask.sum()

        if count > 0:
            valid = df.loc[df[col].le(5000) & df[col].notna(), col]

            cap = round(valid.quantile(0.99), 1) if not valid.empty else 5000

            print(f"{col}: {count} spikes found (max={df.loc[spike_mask, col].max():.0f})")
            print(f"  -> Capping at: {cap}")

            for idx in df.index[spike_mask]:
                audit_log.append({
                    'Record_ID': df.at[idx, 'Record_ID'],
                    'Column': col,
                    'Old': df.at[idx, col],
                    'New': cap,
                    'Rule': 'R8'
                })

            df.loc[spike_mask, col] = cap

    return df



In [77]:
# df = clean_outliers(df)

In [78]:
# C-9. Missing gap filling within allowed substitution rules.
def fill_missing_ids(df):

    # Step 1: extract numeric part
    record_nums = df['Record_ID'].str.replace('E', '').astype(int)

    # Step 2: find missing IDs
    all_expected = set(range(record_nums.min(), record_nums.max() + 1))
    all_actual = set(record_nums)
    missing_ids = sorted(all_expected - all_actual)

    # Step 3: create placeholder rows
    gap_rows = []
    for mid in missing_ids:
        gap_rows.append({
            'Record_ID': f'E{mid:05d}',
            'Plant_ID': np.nan,
            'Stack_ID': np.nan,
            'Flow_Rate_m3_hr': np.nan,
            'TS': pd.NaT,
            'PM2.5': np.nan,
            'SO2': np.nan,
            'NOx': np.nan,
            'Unit': 'ug/m3',
            'Status': 'GAP_FILLED',
            'Lat_Lon': np.nan
        })

    # Step 4: insert rows
    if gap_rows:
        df = pd.concat([df, pd.DataFrame(gap_rows)], ignore_index=True)

    # Step 5: sort
    df = df.sort_values('Record_ID').reset_index(drop=True)

    # Step 6: print summary
    print(f'Inserted {len(gap_rows)} placeholder rows')
    print(f'Total rows now: {len(df)}')
    print('Status counts:')
    print(df['Status'].value_counts(dropna=False))

    return df

In [79]:
def fill_missing_values(df):
    df = df.copy()

    # create flag columns
    for col in ['PM2.5', 'SO2', 'NOx']:
        df[col + '_Filled'] = False

    count = 0

    # sort data (important)
    df = df.sort_values(['Plant_ID', 'Stack_ID', 'TS'])

    for col in ['PM2.5', 'SO2', 'NOx']:

        for i in range(1, len(df)-1):

            val = df.iloc[i][col]
            status = df.iloc[i]['Status']

            if pd.isna(val) and status in ['OK', 'UNKNOWN']:

                prev_val = df.iloc[i-1][col]
                next_val = df.iloc[i+1][col]

                # same sensor check
                same_sensor = (
                    df.iloc[i]['Plant_ID'] == df.iloc[i-1]['Plant_ID'] ==
                    df.iloc[i+1]['Plant_ID'] and
                    df.iloc[i]['Stack_ID'] == df.iloc[i-1]['Stack_ID'] ==
                    df.iloc[i+1]['Stack_ID']
                )

                if same_sensor and pd.notna(prev_val) and pd.notna(next_val):

                    new_val = (prev_val + next_val) / 2

                    df.at[df.index[i], col] = new_val
                    df.at[df.index[i], col + '_Filled'] = True

                    count += 1

    print("Total values filled:", count)

    return df

In [80]:
# df = fill_missing_ids(df)

In [81]:
# df = fill_missing_values(df)

In [82]:
#10
#def apply_maintenance(df, maint):
 #   df = df.copy()

  #  maint['Maint_Start'] = pd.to_datetime(maint['Maint_Start'])
   # maint['Maint_End'] = pd.to_datetime(maint['Maint_End'])
    #df['TS'] = pd.to_datetime(df['TS'], dayfirst=True, errors='coerce')

    #count = 0

    #for i in maint.index:
     #   plant = maint.at[i, 'Plant_ID']
      #  stack = maint.at[i, 'Stack_ID']
       # start = maint.at[i, 'Maint_Start']
        #end = maint.at[i, 'Maint_End']

        # create condition (simple)
        #mask = (
         #   (df['Plant_ID'] == plant) &
          #  (df['Stack_ID'] == stack) &
           # (df['TS'] >= start) &
            #(df['TS'] <= end)
        #)

        #affected = mask.sum()

        #if affected > 0:
         #   df.loc[mask, ['PM2.5', 'SO2', 'NOx']] = None
          #  df.loc[mask, 'Status'] = 'MAINT'

           # count += affected

    #print("Rows affected by maintenance:", count)

    #return df
def apply_maintenance(df, maint):
    global audit_log

    df = df.copy()

    maint['Maint_Start'] = pd.to_datetime(maint['Maint_Start'])
    maint['Maint_End'] = pd.to_datetime(maint['Maint_End'])
    df['TS'] = pd.to_datetime(df['TS'], dayfirst=True, errors='coerce')

    count = 0

    for i in maint.index:
        plant = maint.at[i, 'Plant_ID']
        stack = maint.at[i, 'Stack_ID']
        start = maint.at[i, 'Maint_Start']
        end = maint.at[i, 'Maint_End']

        mask = (
            (df['Plant_ID'] == plant) &
            (df['Stack_ID'] == stack) &
            (df['TS'] >= start) &
            (df['TS'] <= end)
        )

        for idx in df[mask].index:

            # log pollutant overwrite
            for col in ['PM2.5', 'SO2', 'NOx']:
                old_val = df.at[idx, col]

                if pd.notna(old_val):
                    audit_log.append({
                        'Record_ID': df.at[idx, 'Record_ID'],
                        'Column': col,
                        'Old': old_val,
                        'New': None,
                        'Rule': 'R10'
                    })

                    df.at[idx, col] = None

            # log status change
            old_status = df.at[idx, 'Status']
            if old_status != 'MAINT':
                audit_log.append({
                    'Record_ID': df.at[idx, 'Record_ID'],
                    'Column': 'Status',
                    'Old': old_status,
                    'New': 'MAINT',
                    'Rule': 'R10'
                })

                df.at[idx, 'Status'] = 'MAINT'

            count += 1

    print("Rows affected by maintenance:", count)

    return df


In [83]:
# df = apply_maintenance(df,maint)

In [84]:
# C-11. LOD handling (below LOD → half LOD)
# What BDL values exist? ── “below detection limit” values are not numeric. They appear like:

# "BDL"
# "<2"
# "<5"

for col in ['PM2.5', 'SO2', 'NOx']:

    count = 0
    examples = []

    for val in df[col]:
        val_str = str(val).strip().upper()

        if val_str == 'BDL' or val_str.startswith('<'):
            count += 1

            if len(examples) < 5:
                examples.append(val)

    print(col, ":", count, "BDL values (e.g.", examples, ")")


PM2.5 : 1391 BDL values (e.g. ['<2.0', '< 5.0', '<2.0', '< 5.0', 'BDL'] )
SO2 : 1386 BDL values (e.g. ['< 4.0', 'BDL', '< 5.0', '<4.0', '<5.0'] )
NOx : 1393 BDL values (e.g. ['< 5.0', 'BDL', '<2.0', 'BDL', '< 5.0'] )


In [85]:
# CLEAN: Replace BDL with LOD/2 ──
# First, build LOD lookup from sensor_master
# (Plant_ID, Stack_ID) -> {PM2.5: lod, SO2: lod, NOx: lod}

# eg: ('PL-01', 'S-01') → {'PM2.5': 2.0, 'SO2': 4.0, 'NOx': 5.0}


#def replace_bdl_with_lod(df, sensors):
 #   df = df.copy()

  #  print("Replacing BDL values...\n")

   # for col in ['PM2.5', 'SO2', 'NOx']:
    #    count = 0

     #   for i in df.index:
            #if df.at[i, 'Status'] == 'MAINT':
                #continue
             
      #      val = str(df.at[i, col]).strip().upper()

            # check BDL or <value
       #     if val == 'BDL' or val.startswith('<'):

        #        plant = df.at[i, 'Plant_ID']
         #       stack = df.at[i, 'Stack_ID']

                # find matching row in sensors
          #      sensor_row = sensors[
           #         (sensors['Plant_ID'] == plant) &
            #        (sensors['Stack_ID'] == stack)
              #  ]

             #   if not sensor_row.empty:
               #     if col == 'PM2.5':
                #        lod = sensor_row.iloc[0]['LOD_PM25']
                 #   elif col == 'SO2':
                  #      lod = sensor_row.iloc[0]['LOD_SO2']
                   # else:
                    #    lod = sensor_row.iloc[0]['LOD_NOx']
                #else:
                 #   lod = 2.0   # default

                # replace with LOD/2
                #df.at[i, col] = round(lod / 2, 2)

                #count += 1

        #print(col, ":", count, "values replaced with LOD/2")

    #return df
def replace_bdl_with_lod(df, sensors):
    global audit_log

    df = df.copy()

    print("Replacing BDL values...\n")

    for col in ['PM2.5', 'SO2', 'NOx']:
        count = 0

        for i in df.index:
            val = str(df.at[i, col]).strip().upper()

            # check BDL or <value
            if val == 'BDL' or val.startswith('<'):

                old_val = df.at[i, col]

                plant = df.at[i, 'Plant_ID']
                stack = df.at[i, 'Stack_ID']

                # find matching row in sensors
                sensor_row = sensors[
                    (sensors['Plant_ID'] == plant) &
                    (sensors['Stack_ID'] == stack)
                ]

                if not sensor_row.empty:
                    if col == 'PM2.5':
                        lod = sensor_row.iloc[0]['LOD_PM25']
                    elif col == 'SO2':
                        lod = sensor_row.iloc[0]['LOD_SO2']
                    else:
                        lod = sensor_row.iloc[0]['LOD_NOx']
                else:
                    lod = 2.0   # default

                new_val = round(lod / 2, 2)

                # 🔥 AUDIT LOG
                audit_log.append({
                    'Record_ID': df.at[i, 'Record_ID'],
                    'Column': col,
                    'Old': old_val,
                    'New': new_val,
                    'Rule': 'R11'
                })

                # replace value
                df.at[i, col] = new_val

                count += 1

        print(col, ":", count, "values replaced with LOD/2")

    return df


In [86]:
# df = replace_bdl_with_lod(df,sensors)

In [87]:
#C-12
def normalize_status(df):
    df = df.copy()

    # convert everything to string first
    df['Status'] = df['Status'].astype(str).str.upper()

    # handle NaN properly
    df['Status'] = df['Status'].replace(['NAN', 'NONE', ''], 'UNKNOWN')

    valid = ['OK', 'FAULT', 'MAINT', 'DOWN', 'OFFLINE', 'ERROR', 'GAP_FILLED', 'UNKNOWN']

    df['Status'] = df['Status'].where(df['Status'].isin(valid), 'UNKNOWN')

    return df

In [88]:
# C-13 # Check every (Plant_ID, Stack_ID) exists in sensor_master


def validate_mapping(df, sensors):
    df = df.copy()

    # merge on Plant_ID and Stack_ID
    merged = df.merge(
        sensors[['Plant_ID', 'Stack_ID']],
        on=['Plant_ID', 'Stack_ID'],
        how='left',
        indicator=True
    )

    return merged

In [89]:
# find invalid rows
merged = validate_mapping(df, sensors)

invalid_rows = merged[merged['_merge'] == 'left_only']

print("Invalid rows:", len(invalid_rows))
print(invalid_rows[['Plant_ID', 'Stack_ID']].drop_duplicates())

Invalid rows: 0
Empty DataFrame
Columns: [Plant_ID, Stack_ID]
Index: []


In [90]:
# C-14. Unit conversion for mg/Nm3 ↔ ug/m3 when needed.
# C-14. Unit conversion mg/Nm3 → ug/m3
def convert_units(df):
    df = df.copy()

    mask = df["Unit"].str.lower() == "mg/nm3"

    for col in ["PM2.5", "SO2", "NOx"]:
        if col in df.columns:
            df.loc[mask, col] = pd.to_numeric(df.loc[mask, col], errors="coerce") * 1000

    df.loc[mask, "Unit"] = "ug/m3"

    return df


In [91]:
# df = convert_units(df)


In [92]:
df

,Record_ID,Plant_ID,Stack_ID,Flow_Rate_m3_hr,TS,PM2.5,SO2,NOx,Unit,Status,Lat_Lon
0,E00001,PL-01,S-01,2033.5,01-01-2025 00:00,170.3,119.3,103.4,ug/m3,down,"13.083,80.2702"
1,E00002,PL-01,S-02,3377.9,01-01-2025 00:00,<2.0,94.2,31.6,µg/m³,MAINT,"13.0839,80.2721"
2,E00003,PL-02,S-01,3798.4,01-01-2025 00:00,123.9,NaN,101.3,mg/Nm3,FAULT,"19.0763,72.8777"
3,E00004,PL-03,S-01,3121.2,01-01-2025 00:00,117.8,147.1,100.8,ug/m3,OK,"12.9719,77.5955"
4,E00005,PL-03,S-02,3700.5,31-12-2024 24:00,37.0,15.0,157.1,mg/Nm3,FAULT,"12.9722,77.5941"
...,...,...,...,...,...,...,...,...,...,...,...
27116,E28795,PL-03,S-02,1341.2,30-01-2025 23:45,168.3,107.7,148.2,µg/m³,FAULT,"12.9713,77.594"
27117,E28796,PL-03,S-03,2725.5,30-01-2025 23:45,34.1,182.4,NaN,ug/m3,MAINT,"12.9709,77.5956"
27118,E28797,LOC-DEL-01,A-01,0.0,30-01-2025 23:45,-9.8,139.6,104.9,ug/m3,FAULT,"28.614,77.2084"
27119,E28798,LOC-DEL-02,A-02,0.0,30-01-2025 23:45,157.9,184.4,20.0,ug/m3,error,"28.6209,77.2156"


In [93]:
# C-15. Ambient vs stack data source tagging.
def add_source_type(df, sensors):
    df = df.copy()

    # remove old column if exists
    if 'Source_Type' in df.columns:
        df = df.drop(columns=['Source_Type'])

    source_info = sensors[['Plant_ID', 'Stack_ID', 'Source_Type']]

    df = df.merge(source_info, on=['Plant_ID', 'Stack_ID'], how='left')

    df['Source_Type'] = df['Source_Type'].fillna('Unknown')

    return df

In [94]:
# df = add_source_type(df, sensors)

# print(df['Source_Type'].value_counts())

In [95]:
# C-16. Lab result digitization QC (double entry checks).
def qc_double_entry(manual):
    manual = manual.copy()

    # Step 1: Calculate % difference
    manual['Diff_Pct'] = abs(
        manual['Lab_PM25_Entry1'] - manual['Lab_PM25_Entry2']
    ) / manual['Lab_PM25_Entry1'] * 100

    # Step 2: Assign QC status
    manual['QC_Status'] = manual['Diff_Pct'].apply(
        lambda d: 'QC_FAIL' if d > 1.0 else 'QC_PASS'
    )

    # Step 3: Print summary
    print("Double-entry QC results:")
    print(manual[['Log_ID', 'Lab_PM25_Entry1', 'Lab_PM25_Entry2', 'Diff_Pct', 'QC_Status']])

    print("\nQC Failures:", (manual["QC_Status"] == "QC_FAIL").sum())

    return manual

In [96]:
# manual = qc_double_entry(manual)

In [97]:
# C-18. PII removal in inspection notes.
import re

def remove_pii_notes(manual):
    manual = manual.copy()

    count = 0

    for i in manual.index:
        note = str(manual.at[i, 'Inspection_Notes'])
        old = note

        # phone
        note = re.sub(r'\d{10}', '[PHONE]', note)

        # email
        note = re.sub(r'\S+@\S+', '[EMAIL]', note)

        if note != old:
            manual.at[i, 'Inspection_Notes'] = note
            count += 1

    print("Notes cleaned:", count)

    return manual

In [98]:
# C-19. Regulatory threshold joins for each pollutant.
def flag_exceedance(df, thresholds):
    df = df.copy()

    # Step 1: convert to numbers
    for col in ['PM2.5', 'SO2', 'NOx']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Step 2: print thresholds
    print("Regulatory thresholds:")
    print(thresholds)
    print()

    # Step 3: default flag
    df['Exceedance_Flag'] = 'OK'

    count = 0

    # Step 4: loop through data
    for i in df.index:
        src = df.at[i, 'Source_Type']

        for col in ['PM2.5', 'SO2', 'NOx']:
            val = df.at[i, col]

            if pd.isna(val) or src == 'Unknown':
                continue

            # find limit from thresholds table
            row = thresholds[
                (thresholds['Pollutant'] == col) &
                (thresholds['Source_Type'] == src)
            ]

            if not row.empty:
                limit = row.iloc[0]['Legal_Limit_ugm3']

                if val > limit:
                    df.at[i, 'Exceedance_Flag'] = 'EXCEEDANCE'
                    count += 1

    # Step 5: print result
    print("Exceedances flagged:", count)
    print(df['Exceedance_Flag'].value_counts())

    return df

In [99]:
# df = flag_exceedance(df, thresholds)

In [100]:
# R1
df = clean_timestamp(df)
audit_log.append({'Rule': 'R1', 'Step': 'Timestamp cleaned'})

# R17
df = convert_utc_to_ist(df)
audit_log.append({'Rule': 'R17', 'Step': 'UTC converted to IST'})

# R2
df = clean_unit(df)
audit_log.append({'Rule': 'R2', 'Step': 'Unit standardized to ug/m3'})

# R3
df = clean_status(df)
audit_log.append({'Rule': 'R3', 'Step': 'Status standardized'})

# R12 (ADD THIS)
df = normalize_status(df)
audit_log.append({'Rule': 'R12', 'Step': 'Status normalized'})

# R4
df = clean_coordinates(df)
audit_log.append({'Rule': 'R4', 'Step': 'Coordinates parsed into Lat/Lon'})

# R5 (if you have)
df = clean_pollutants(df)
audit_log.append({'Rule': 'R5', 'Step': 'Negative values handled'})

# R6
df = remove_duplicates(df)
audit_log.append({'Rule': 'R6', 'Step': 'Duplicates removed'})

# R7 (if calibration exists)
df = apply_calibration(df, sensors)
audit_log.append({'Rule': 'R7', 'Step': 'Calibration applied'})



# R9
df = fill_missing_ids(df)
df = fill_missing_values(df)
audit_log.append({'Rule': 'R9', 'Step': 'Missing values filled'})


df = remove_duplicates(df)

def fix_existing_maint(df):
    global audit_log

    df = df.copy()

    mask = df['Status'] == 'MAINT'

    for idx in df[mask].index:
        for col in ['PM2.5', 'SO2', 'NOx']:
            old_val = df.at[idx, col]

            if pd.notna(old_val):
                audit_log.append({
                    'Record_ID': df.at[idx, 'Record_ID'],
                    'Column': col,
                    'Old': old_val,
                    'New': None,
                    'Rule': 'R10'
                })

                df.at[idx, col] = None

    print("Existing MAINT rows fixed:", mask.sum())

    return df
df = fix_existing_maint(df)

# R10
df = apply_maintenance(df, maint)
audit_log.append({'Rule': 'R10', 'Step': 'Maintenance downtime applied'})

# R11
df = replace_bdl_with_lod(df, sensors)
audit_log.append({'Rule': 'R11', 'Step': 'BDL replaced with LOD/2'})

df = clean_outliers(df)
audit_log.append({'Rule': 'R8', 'Step': 'Outliers cleaned (pass 1)'})

# R13 (validation only)
audit_log.append({'Rule': 'R13', 'Step': 'Plant-Stack mapping validated'})

# R14
df = convert_units(df)
audit_log.append({'Rule': 'R14', 'Step': 'Final unit check (ug/m3)'})

# R8
df = clean_outliers(df)
audit_log.append({'Rule': 'R8', 'Step': 'Outliers capped using 99th percentile(pass 2)'})

# R15
df = add_source_type(df, sensors)
audit_log.append({'Rule': 'R15', 'Step': 'Source_Type added'})

# R16
manual = qc_double_entry(manual)
audit_log.append({'Rule': 'R16', 'Step': 'Manual QC applied'})



# R18
manual = remove_pii_notes(manual)
audit_log.append({'Rule': 'R18', 'Step': 'PII removed from notes'})

# R19
df = flag_exceedance(df, threshold)
audit_log.append({'Rule': 'R19', 'Step': 'Exceedance flags added'})

df = clean_outliers(df)
audit_log.append({'Rule': 'R8', 'Step': 'Outliers cleaned (final pass)'})


Slash fixed: 1329
ISO fixed: 545
Midnight fixed: 797
UTC converted to IST: 1292
BEFORE — Total rows: 27121
AFTER  — Total rows: 26385
Duplicates removed: 736
Inserted 2415 placeholder rows
Total rows now: 28800
Status counts:
Status
OK            8313
MAINT         7795
FAULT         6718
GAP_FILLED    2415
ERROR         1186
OFFLINE       1136
DOWN          1117
UNKNOWN        120
Name: count, dtype: int64
Total values filled: 0
BEFORE — Total rows: 28800
AFTER  — Total rows: 26386
Duplicates removed: 2414
Existing MAINT rows fixed: 7795


C:\Users\hasin\AppData\Local\Temp\ipykernel_14536\3362113657.py:41: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  maint['Maint_Start'] = pd.to_datetime(maint['Maint_Start'])
C:\Users\hasin\AppData\Local\Temp\ipykernel_14536\3362113657.py:42: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  maint['Maint_End'] = pd.to_datetime(maint['Maint_End'])


Rows affected by maintenance: 296
Replacing BDL values...

PM2.5 : 0 values replaced with LOD/2
SO2 : 0 values replaced with LOD/2
NOx : 0 values replaced with LOD/2
PM2.5: 546 spikes found (max=10203)
  -> Capping at: 409.6
SO2: 600 spikes found (max=10203)
  -> Capping at: 448.7
NOx: 568 spikes found (max=10203)
  -> Capping at: 466.1
PM2.5: 1867 spikes found (max=409600)
  -> Capping at: 409.6
SO2: 1867 spikes found (max=448731)
  -> Capping at: 448.7
NOx: 1867 spikes found (max=466100)
  -> Capping at: 466.1
Double-entry QC results:
   Log_ID  Lab_PM25_Entry1  Lab_PM25_Entry2    Diff_Pct QC_Status
0   L0001             98.7            987.0  900.000000   QC_FAIL
1   L0002            156.4            156.7    0.191816   QC_PASS
2   L0003             49.6             49.2    0.806452   QC_PASS
3   L0004            123.7            124.2    0.404204   QC_PASS
4   L0005            132.9            132.8    0.075245   QC_PASS
5   L0006             69.9            699.0  900.000000   QC_

In [101]:
# C-20. Audit trail creation for corrections

#remove unwanted columns
df.drop(columns=["Lat_Lon","Sector","Zero_Drift","Span_Mult","LOD_PM25","LOD_SO2","LOD_NOx","PM2.5_Filled","SO2_Filled","NOx_Filled"],inplace=True) 
df.to_csv("raw_cems_data_cleaned.csv", index=False)


audit_df = pd.DataFrame(audit_log)
audit_df.to_csv("cleaning_log.csv", index=False)

print("Cleaned data saved:", len(df))
print("Audit log saved:", len(audit_df))

Cleaned data saved: 26386
Audit log saved: 61486


In [102]:
# R20
audit_log.append({'Rule': 'R20', 'Step': 'Audit log created'})

In [103]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26386 entries, 0 to 26385
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Record_ID        26386 non-null  str           
 1   Plant_ID         26385 non-null  object        
 2   Stack_ID         26385 non-null  object        
 3   Flow_Rate_m3_hr  21067 non-null  float64       
 4   TS               26385 non-null  datetime64[us]
 5   PM2.5            18391 non-null  float64       
 6   SO2              18391 non-null  float64       
 7   NOx              18391 non-null  float64       
 8   Unit             26386 non-null  str           
 9   Status           26386 non-null  str           
 10  Lat              26385 non-null  float64       
 11  Lon              26385 non-null  float64       
 12  Source_Type      26386 non-null  str           
 13  Exceedance_Flag  26386 non-null  str           
dtypes: datetime64[us](1), float64(6), object(2), str(

In [104]:
import os

OUT_DIR = '../cleaned'

os.makedirs(OUT_DIR, exist_ok=True)

In [105]:
df.to_csv(f'{OUT_DIR}/cleaned_data.csv', index=False)

manual.to_csv(f'{OUT_DIR}/manual_cleaned.csv', index=False)

maint.to_csv(f'{OUT_DIR}/maintenance_cleaned.csv', index=False)

sensors.to_csv(f'{OUT_DIR}/sensor_master_cleaned.csv', index=False)

threshold.to_csv(f'{OUT_DIR}/thresholds_cleaned.csv', index=False)

audit_df = pd.DataFrame(audit_log)
audit_df.to_csv(f'{OUT_DIR}/cleaning_log.csv', index=False)